In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    logging,
    BitsAndBytesConfig
)
from tqdm import tqdm
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score, classification_report
from utils.preprocessing import prepare_dataset, tokenize_function 
from utils.evaluation import compute_metrics, evaluate_trainer

try:
    import bitsandbytes as bnb
    USE_QUANTIZATION = True
except ImportError:
    USE_QUANTIZATION = False
    print("bitsandbytes не найден. Обучение в FP16 (требуется ~14-16 ГБ VRAM).")

In [4]:
logging.set_verbosity_error()
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization_config = None
if USE_QUANTIZATION:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        llm_int8_enable_fp32_cpu_offload=True 
    )
    print("Активировано 4-битное квантование")

def load_qwen_model():
    model_kwargs = {
        "pretrained_model_name_or_path": model_name,
        "num_labels": 3,
        "problem_type": "single_label_classification",
        "ignore_mismatched_sizes": True,
        "torch_dtype": torch.float16 if not USE_QUANTIZATION else None,
    }

    if USE_QUANTIZATION:
        model_kwargs["quantization_config"] = quantization_config
        model_kwargs["device_map"] = None 
    else:
        model_kwargs["device_map"] = None #

    model = AutoModelForSequenceClassification.from_pretrained(**model_kwargs)
    
    if not USE_QUANTIZATION:
        model.to(device)
    else:
        model = prepare_model_for_kbit_training(model)
        
    return model

Активировано 4-битное квантование


## Preprocessing

In [5]:
df_train = pd.read_csv("lemmatized\\train_lemmatized.csv")
df_val = pd.read_csv("lemmatized\\validation_lemmatized.csv")
df_test = pd.read_csv("lemmatized\\test_lemmatized.csv")
df_expert_45 = pd.read_csv("lemmatized\\expert_core_45_lemmatized.csv")
df_expert_90 = pd.read_csv("lemmatized\\expert_core_90_lemmatized.csv")
df_expert_180 = pd.read_csv("lemmatized\\expert_core_180_lemmatized.csv")

samples_per_class = 1000
dfs = []
for label in df_test['label'].unique():
    class_df = df_test[df_test['label'] == label]
    n = min(len(class_df), samples_per_class)
    sampled = class_df.sample(n=n, random_state=42)
    dfs.append(sampled)
df_test = pd.concat(dfs).reset_index(drop=True)

dataset_train = prepare_dataset(df_train)
dataset_val = prepare_dataset(df_val)
dataset_test = prepare_dataset(df_test)
dataset_expert_45 = prepare_dataset(df_expert_45)
dataset_expert_90 = prepare_dataset(df_expert_90)
dataset_expert_180 = prepare_dataset(df_expert_180)

map_func = lambda examples: tokenize_function(examples, tokenizer)
tokenized_train = dataset_train.map(map_func, remove_columns=['text'], batched=True)
tokenized_val = dataset_val.map(map_func, remove_columns=['text'], batched=True)
tokenized_test = dataset_test.map(map_func, remove_columns=['text'], batched=True)
tokenized_expert_45 = dataset_expert_45.map(map_func, remove_columns=['text'], batched=True)
tokenized_expert_90 = dataset_expert_90.map(map_func, remove_columns=['text'], batched=True)
tokenized_expert_180 = dataset_expert_180.map(map_func, remove_columns=['text'], batched=True)

Map: 100%|██████████| 180/180 [00:00<00:00, 9060.60 examples/s]


## Baseline

In [ ]:
model_zs = load_qwen_model()
model_zs.eval()

if model_zs.config.pad_token_id is None:
    model_zs.config.pad_token_id = tokenizer.pad_token_id

test_dataset_formatted = tokenized_test.with_format("torch", columns=["input_ids", "attention_mask", "labels"])
dataloader = DataLoader(test_dataset_formatted, batch_size=32, shuffle=False)

Loading weights: 100%|██████████| 338/338 [00:03<00:00, 92.82it/s, Materializing param=model.norm.weight]                               


In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference"):
        input_ids = batch["input_ids"].to(model_zs.device)
        attention_mask = batch["attention_mask"].to(model_zs.device)
        labels = batch["labels"]
        
        outputs = model_zs(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

In [ ]:
acc = accuracy_score(all_labels, all_preds)
f1_macro = f1_score(all_labels, all_preds, average='macro')

print(f"\nРезультаты модели: Zero-shot Qwen")
print(f"   Accuracy:     {acc:.4f}")
print(f"   F1 Macro:     {f1_macro:.4f}")

"""Результаты модели: Zero-shot Qwen
   Accuracy:     0.2787
   F1 Macro:     0.2122"""

In [6]:
def run_experiment(train_ds, name, epochs=10, batch_size=4, lr=1e-4):
    print(f"\nОбучение Qwen LoRA: {name} примеров")
    
    model = load_qwen_model()
    
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
        if hasattr(model, 'config') and hasattr(model.config, 'eos_token_id'):
             pass 
    
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"]
    )
    
    try:
        model = get_peft_model(model, peft_config)
    except ValueError:
        peft_config.target_modules = "all-linear"
        model = load_qwen_model()
        if model.config.pad_token_id is None:
            model.config.pad_token_id = tokenizer.pad_token_id
        model = get_peft_model(model, peft_config)
        
    model.print_trainable_parameters()
    
    args = TrainingArguments(
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=0.01,
        report_to="none",
        fp16=not USE_QUANTIZATION,
        seed=42,
        save_strategy="no",
        optim="paged_adamw_8bit" if USE_QUANTIZATION else "adamw_torch",
        logging_steps=20
    )
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    return trainer

In [ ]:
trainer_lora_45 = run_experiment(tokenized_expert_45, "45", epochs=30, batch_size=5)


Обучение Qwen LoRA: 45 примеров


Loading weights: 100%|██████████| 338/338 [00:03<00:00, 89.59it/s, Materializing param=model.norm.weight]                               


trainable params: 5,057,024 || all params: 7,075,686,912 || trainable%: 0.0715
{'loss': '1.992', 'grad_norm': '33.32', 'learning_rate': '9.296e-05', 'epoch': '2.222'}
{'loss': '0.3025', 'grad_norm': '8.168', 'learning_rate': '8.556e-05', 'epoch': '4.444'}
{'loss': '0.01412', 'grad_norm': '0.008978', 'learning_rate': '7.815e-05', 'epoch': '6.667'}
{'loss': '5.076e-05', 'grad_norm': '0.01616', 'learning_rate': '7.074e-05', 'epoch': '8.889'}
{'loss': '1.061e-05', 'grad_norm': '0.006831', 'learning_rate': '6.333e-05', 'epoch': '11.11'}
{'loss': '6.213e-06', 'grad_norm': '0.004035', 'learning_rate': '5.593e-05', 'epoch': '13.33'}
{'loss': '5.182e-06', 'grad_norm': '0.001454', 'learning_rate': '4.852e-05', 'epoch': '15.56'}
{'loss': '5.043e-06', 'grad_norm': '0.001014', 'learning_rate': '4.111e-05', 'epoch': '17.78'}
{'loss': '4.526e-06', 'grad_norm': '0.001545', 'learning_rate': '3.37e-05', 'epoch': '20'}
{'loss': '4.241e-06', 'grad_norm': '0.002254', 'learning_rate': '2.63e-05', 'epoch': '

In [ ]:
res_lora_45 = evaluate_trainer(trainer_lora_45, tokenized_test, "Qwen LoRA (45 examples)")

{'eval_loss': '2.465', 'eval_accuracy': '0.517', 'eval_f1_macro': '0.4809', 'eval_runtime': '96.8', 'eval_samples_per_second': '30.99', 'eval_steps_per_second': '3.874', 'epoch': '30'}

Результаты модели: Qwen LoRA (45 examples)
   Accuracy:     0.5170
   F1 Macro:     0.4809


In [ ]:
trainer_lora_90 = run_experiment(tokenized_expert_90, "90", epochs=30, batch_size=10)


Обучение Qwen LoRA: 90 примеров


Loading weights: 100%|██████████| 338/338 [00:03<00:00, 92.10it/s, Materializing param=model.norm.weight]                               


trainable params: 5,057,024 || all params: 7,075,686,912 || trainable%: 0.0715
{'loss': '1.416', 'grad_norm': '121.2', 'learning_rate': '9.296e-05', 'epoch': '2.222'}
{'loss': '0.211', 'grad_norm': '1.6', 'learning_rate': '8.556e-05', 'epoch': '4.444'}
{'loss': '0.009301', 'grad_norm': '0.05699', 'learning_rate': '7.815e-05', 'epoch': '6.667'}
{'loss': '7.612e-05', 'grad_norm': '0.01068', 'learning_rate': '7.074e-05', 'epoch': '8.889'}
{'loss': '1.052e-05', 'grad_norm': '0.006144', 'learning_rate': '6.333e-05', 'epoch': '11.11'}
{'loss': '6.032e-06', 'grad_norm': '0.00231', 'learning_rate': '5.593e-05', 'epoch': '13.33'}
{'loss': '5.655e-06', 'grad_norm': '0.002008', 'learning_rate': '4.852e-05', 'epoch': '15.56'}
{'loss': '5.6e-06', 'grad_norm': '0.003421', 'learning_rate': '4.111e-05', 'epoch': '17.78'}
{'loss': '5.39e-06', 'grad_norm': '0.003306', 'learning_rate': '3.37e-05', 'epoch': '20'}
{'loss': '5.222e-06', 'grad_norm': '0.001606', 'learning_rate': '2.63e-05', 'epoch': '22.22'}

In [ ]:
res_lora_90 = evaluate_trainer(trainer_lora_90, tokenized_test, "Qwen LoRA (90 examples)")

{'eval_loss': '2.294', 'eval_accuracy': '0.539', 'eval_f1_macro': '0.5369', 'eval_runtime': '101.6', 'eval_samples_per_second': '29.51', 'eval_steps_per_second': '3.689', 'epoch': '30'}

Результаты модели: Qwen LoRA (90 examples)
   Accuracy:     0.5390
   F1 Macro:     0.5369


In [ ]:
trainer_lora_180 = run_experiment(tokenized_expert_180, "180", epochs=30, batch_size=20)


Обучение Qwen LoRA: 180 примеров


Loading weights: 100%|██████████| 338/338 [00:08<00:00, 41.53it/s, Materializing param=model.norm.weight]                               


trainable params: 5,057,024 || all params: 7,075,686,912 || trainable%: 0.0715
{'loss': '1.571', 'grad_norm': '144.5', 'learning_rate': '9.296e-05', 'epoch': '2.222'}
{'loss': '0.3105', 'grad_norm': '30.79', 'learning_rate': '8.556e-05', 'epoch': '4.444'}
{'loss': '0.01465', 'grad_norm': '0.2376', 'learning_rate': '7.815e-05', 'epoch': '6.667'}
{'loss': '0.0001195', 'grad_norm': '0.01722', 'learning_rate': '7.074e-05', 'epoch': '8.889'}
{'loss': '1.909e-05', 'grad_norm': '0.004612', 'learning_rate': '6.333e-05', 'epoch': '11.11'}
{'loss': '1.178e-05', 'grad_norm': '0.001433', 'learning_rate': '5.593e-05', 'epoch': '13.33'}
{'loss': '1.153e-05', 'grad_norm': '0.002911', 'learning_rate': '4.852e-05', 'epoch': '15.56'}
{'loss': '9.729e-06', 'grad_norm': '0.002423', 'learning_rate': '4.111e-05', 'epoch': '17.78'}
{'loss': '9.534e-06', 'grad_norm': '0.002098', 'learning_rate': '3.37e-05', 'epoch': '20'}
{'loss': '9.422e-06', 'grad_norm': '0.002584', 'learning_rate': '2.63e-05', 'epoch': '22

In [8]:
res_lora_180 = evaluate_trainer(trainer_lora_180, tokenized_test, "LoRA (180 examples)")

{'eval_loss': '3.046', 'eval_accuracy': '0.5637', 'eval_f1_macro': '0.4939', 'eval_runtime': '96.24', 'eval_samples_per_second': '31.17', 'eval_steps_per_second': '3.897', 'epoch': '30'}

Результаты модели: LoRA (180 examples)
   Accuracy:     0.5637
   F1 Macro:     0.4939
